In [1]:
!git clone https://github.com/zhao-zilong/ssc-cot.git

Cloning into 'ssc-cot'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 134 (delta 54), reused 130 (delta 50), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 348.81 KiB | 4.20 MiB/s, done.
Resolving deltas: 100% (54/54), done.


In [2]:
!pip install -q peft transformers datasets accelerate bitsandbytes trl

In [3]:
import os
files = os.listdir("/home/workspace/ssc-cot/Dataset")
print(len(files))

100


In [4]:
import os
import json

dataset_path = "/home/workspace/ssc-cot/Dataset"

all_data = []

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.endswith(".json"):
            file_path = os.path.join(root, file)
            with open(file_path, "r", encoding="utf-8") as f:
                try:
                    data = json.load(f)
                    if isinstance(data, list):
                        all_data.extend(data)
                    elif isinstance(data, dict):
                        all_data.append(data)
                except Exception as e:
                    print(f"Error reading {file_path}: {e}")

print(f"Total samples loaded: {len(all_data)}")

Total samples loaded: 100


In [5]:
print(all_data[0])
print(all_data[0].keys())

{'question': 'Simplify tan(100)+4sin(100)', 'original_form': 'tan(100)+4sin(100)', 'intermediate-results': [{'result': 'tan(100) + 4sin(100) = (sin(100)+4sin(100)cos(100))/cos(100)', 'step': 1, 'score': 1, 'branch': 'None', 'branch-level': 'None'}, {'result': 'tan(100) + 4sin(100) = (sin(100)+2sin(200))/cos(100)', 'step': 2, 'score': 2, 'branch': 'None', 'branch-level': 'None'}, {'result': 'tan(100) + 4sin(100) = (sin(100)-2sin(20))/cos(100)', 'step': 3, 'score': 3, 'branch': 'None', 'branch-level': 'None'}, {'result': 'tan(100) + 4sin(100) = (2sin(20)-cos(10))/(sin(10))', 'step': 4, 'score': 4, 'branch': 'None', 'branch-level': 'None'}, {'result': 'tan(100) + 4sin(100) = (-\\sqrt(3)sin(10))/(sin(10))', 'step': 5, 'score': 5, 'branch': 'None', 'branch-level': 'None'}, {'result': 'tan(100) + 4sin(100) = -\\sqrt(3)', 'step': 6, 'score': 6, 'branch': 'None', 'branch-level': 'None'}], 'intermediate-results_alternative': [{'result': 'tan(100) + 4sin(100) = -cot(10) + 4sin(100)', 'step': 1, 

In [6]:
from datasets import Dataset

STEP_SEPARATOR = " ки"
POSITIVE_TOKEN = "+"
NEGATIVE_TOKEN = "-"

def convert_prm(sample):
    question = sample.get("question", "")
    intermediate = sample.get("intermediate-results", [])
    steps = [s.get("result", "") for s in intermediate]
    scores = [s.get("score", 0) for s in intermediate]
    return {
        "question": question,
        "steps": steps,
        "scores": scores
    }

def format_prm_text(sample):
    question = sample["question"]
    steps = sample["steps"]
    scores = sample["scores"]
    text = question + "\n"
    for step, score in zip(steps, scores):
        label = POSITIVE_TOKEN if score > 0 else NEGATIVE_TOKEN
        text += step + STEP_SEPARATOR + label + "\n"
    return {"text": text}

hf_data = [convert_prm(x) for x in all_data if isinstance(x, dict)]
dataset = Dataset.from_list(hf_data)
dataset = dataset.map(format_prm_text)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(dataset)
print(dataset["train"][0]["text"][:500])

/home/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 100/100 [00:00<00:00, 9870.58 examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'steps', 'scores', 'text'],
        num_rows: 90
    })
    test: Dataset({
        features: ['question', 'steps', 'scores', 'text'],
        num_rows: 10
    })
})
In triangle ABC, the angles A, B, and C correspond to the three sides a, b, and c. If 9a^2 + 9b^2 - 19c^2 = 0, then find the value of cotC/(cotA + cotB)
cotC/(cotA + cotB) = (cosC/sinC)/(cosA/sinA + cosB/sinB) ки+
(cosC/sinC)/(cosA/sinA + cosB/sinB) = (cosC/sinC)/((cosAsinB+cosBsinA)/(sinAsinB)) ки+
(cosC/sinC)/((cosAsinB+cosBsinA)/(sinAsinB)) = sinAsinB/((sinC)^2)cosC ки+
sinAsinB/((sinC)^2)cosC = ab/c^2*(a^2+b^2-c^2)/(2ab) ки+
ab/c^2*(a^2+b^2-c^2)/(2ab) = (a^2+b^2-c^2)/(2c^2) ки+
(a^2+b^2-c^2)


In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-Math-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"



In [16]:
MAX_LENGTH = 1024

def tokenize(sample):
    result = tokenizer(
        sample["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_train = dataset["train"].map(tokenize, batched=True, remove_columns=dataset["train"].column_names)
tokenized_eval = dataset["test"].map(tokenize, batched=True, remove_columns=dataset["test"].column_names)

print(tokenized_train)

Map: 100%|██████████| 10/10 [00:00<00:00, 804.28 examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 90
})


In [17]:
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model
import torch

# -------------------------
# 1. Load model in 4-bit (QLoRA)
# -------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0}   # ✅ FIXED (no auto split)
)

# -------------------------
# 2. Enable gradient checkpointing
# -------------------------
model.gradient_checkpointing_enable()

# -------------------------
# 3. Apply LoRA
# -------------------------
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # safer for most 7B models
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# -------------------------
# 4. Data collator
# -------------------------
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# -------------------------
# 5. Training arguments
# -------------------------
training_args = TrainingArguments(
    output_dir="./prm_lora_output",
    num_train_epochs=10,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    learning_rate=1e-4,
    warmup_steps=400,

    fp16=True,

    eval_strategy="steps",
    eval_steps=100,

    save_strategy="steps",
    save_steps=100,

    logging_steps=10,
    report_to="none",

    dataloader_pin_memory=False,

    optim="paged_adamw_8bit"   # ✅ memory efficient optimizer
)

# -------------------------
# 6. Trainer
# -------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator
)

# -------------------------
# 7. Train
# -------------------------
torch.cuda.empty_cache()
trainer.train()

Loading weights:   1%|          | 2/339 [00:00<00:29, 11.43it/s]/home/.venv/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 339/339 [00:01<00:00, 227.97it/s]


Step,Training Loss,Validation Loss
60,1.162804,1.182262


TrainOutput(global_step=60, training_loss=1.1931248664855958, metrics={'train_runtime': 507.2421, 'train_samples_per_second': 1.774, 'train_steps_per_second': 0.118, 'total_flos': 3.91255994400768e+16, 'train_loss': 1.1931248664855958, 'epoch': 10.0})

In [18]:
model.save_pretrained("./prm_lora_final_2")
tokenizer.save_pretrained("./prm_lora_final_2")
print("Model saved to ./prm_lora_final_2")

Model saved to ./prm_lora_final_2
